# 6.12 · t-SNE / t-Distributed Stochastic Neighbor Embedding

> **课程定位 / Where this fits**
> 第 12 课，**Part 6 · 无监督学习**。
> Lesson 12, **Part 6 · Unsupervised Learning**.
>
> PCA(6.8)是线性、保全局的降维。t-SNE 是**非线性、保局部邻域**的**可视化**专用降维：把高维数据（如 784 维 MNIST）摊成一张 2D 图，让同类点聚成肉眼可见的簇。它几乎是高维数据探索的标准工具——但有一堆**极易被误读**的坑（簇大小、簇间距离都不可信）。
> PCA (6.8) is linear, global DR. t-SNE is **nonlinear, neighbor-preserving** DR for **visualization**: it lays high-dim data (e.g. 784-D MNIST) flat into 2-D so similar points form visible clusters. It's almost the standard tool for high-dim exploration — but full of **easily-misread** pitfalls (cluster sizes and distances are not trustworthy).

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $p_{ij}$ —— 高维空间里点 $i,j$ 的相似度（高斯）/ high-dim similarity (Gaussian)
> - $q_{ij}$ —— 低维空间里的相似度（t 分布）/ low-dim similarity (t-distribution)
> - perplexity —— 每个点的"有效邻居数" / effective number of neighbors per point
> - KL —— KL 散度（衡量两个分布的差异）/ KL divergence

> 💡 **面试相关 / Interview-relevant**
> - "t-SNE 的原理（概率相似度 + KL 散度）"（★★★★★）
> - "为什么低维用 t 分布（重尾）解决拥挤问题"（★★★★★）
> - "perplexity 是什么 / 怎么影响结果"（★★★★★）
> - "t-SNE 结果的哪些方面不可信"（★★★★★，簇大小/距离/全局）
> - "t-SNE 为什么不能 transform 新数据" / "vs PCA vs UMAP"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解高维/低维相似度（高斯 vs t 分布）+ KL 目标。
   Understand high/low-dim similarities (Gaussian vs t) and the KL objective.
2. 理解拥挤问题与 t 分布重尾的作用。
   Understand the crowding problem and the role of the t-distribution's heavy tail.
3. 理解 perplexity 的影响。
   Understand the effect of perplexity.
4. 学会正确解读 + 避开常见误区。
   Read results correctly and avoid common pitfalls.
5. 知道 t-SNE 的局限（不可 transform、慢、随机）。
   Know t-SNE's limits (no transform, slow, stochastic).

## 目录 / TOC
1. [先建直觉：把邻居关系搬到 2D](#1)
2. [原理：相似度 + KL ⭐](#2)
3. [🔢 数据：Digits + t-SNE ⭐](#3)
4. [perplexity 的影响 ⭐](#4)
5. [误区：什么不可信 ⭐](#5)
6. [小结](#6)


<a id="1"></a>
## 1. 先建直觉：把邻居关系搬到 2D / Intuition First

我们想把几百维的数据画成一张 2D 图给人看。PCA 是"找一个最好的拍照角度"（线性投影），但高维里缠绕在一起的结构，换个角度拍还是缠在一起。
We want to draw high-dim data as a 2-D picture for humans. PCA is "find the best camera angle" (a linear projection), but structure tangled in high-dim stays tangled from any angle.

t-SNE 换了个思路：**不在乎绝对位置，只在乎"谁和谁是邻居"。** 它先在高维空间量出"每个点的近邻是谁、有多近"，然后在 2D 平面上摆放这些点，**努力让 2D 里的邻居关系尽量还原高维里的邻居关系**。结果就是：高维里靠得近的点，在 2D 图上也聚在一起 → 同类自然成簇。
t-SNE takes a different view: **it ignores absolute positions and only cares "who is whose neighbor".** It first measures, in high-dim, each point's neighbors and how close they are; then it places points on a 2-D plane, **trying to reproduce the high-dim neighbor relationships in 2-D**. The result: points close in high-dim end up close in 2-D → same-class points cluster naturally.

代价是：为了把局部邻居关系摆好，它会**牺牲全局信息**——所以簇之间的距离、簇的大小都别当真（第 5 节细讲）。
The price: to lay out local neighborhoods well, it **sacrifices global information** — so don't trust between-cluster distances or cluster sizes (detailed in Section 5).


<a id="2"></a>
## 2. 原理：相似度 + KL ⭐ / Similarities & KL Divergence

t-SNE 把"保持邻居关系"形式化成**让两个概率分布对齐**：
t-SNE formalizes "preserve neighborhoods" as **aligning two probability distributions**:

1. **高维相似度 $p_{ij}$**：用高斯核把点对距离转成"点 $i$ 选 $j$ 当邻居"的概率——越近概率越大。每个点的高斯有多宽，由 **perplexity** 决定（相当于"看几个邻居"）。
   **High-dim similarity $p_{ij}$:** a Gaussian kernel turns pairwise distances into "probability that $i$ picks $j$ as a neighbor" — closer = higher. Each point's Gaussian width is set by **perplexity** (≈ "how many neighbors to consider").
2. **低维相似度 $q_{ij}$**：在 2D 嵌入里，用 **t 分布（自由度 1，即柯西分布）** 计算相似度。
   **Low-dim similarity $q_{ij}$:** in the 2-D embedding, use a **t-distribution (1 dof, i.e. Cauchy)** for similarity.
3. **目标**：最小化两个分布的 **KL 散度** $\mathrm{KL}(P\|Q)=\sum_{ij}p_{ij}\log\frac{p_{ij}}{q_{ij}}$，用梯度下降不断挪动 2D 里的点。
   **Objective:** minimize the **KL divergence** $\mathrm{KL}(P\|Q)$, moving the 2-D points by gradient descent.

**为什么低维要用 t 分布（重尾）—— 拥挤问题(crowding)**：高维空间的"体积"远大于 2D，硬把所有邻居塞进 2D 会过度拥挤。t 分布的**重尾**让中等距离的点对在低维**多占一点空间**（允许它们被推得稍远），缓解拥挤，让簇能分得开。而 KL 的不对称性让 t-SNE **优先保住近邻**（把近的放近），不太在意远的。
**Why a heavy-tailed t-distribution in low-dim — the crowding problem:** high-dim "volume" vastly exceeds 2-D, so cramming all neighbors into 2-D overcrowds. The t-distribution's **heavy tail** gives medium-distance pairs **more room** in low-dim (lets them be pushed a bit apart), easing crowding so clusters separate. KL's asymmetry makes t-SNE **prioritize keeping near points near**, caring little about far ones.


<a id="3"></a>
## 3. 数据：Digits + t-SNE ⭐ / Digits & t-SNE

**Digits**（64 维手写数字，MNIST 的迷你版）。先用 PCA 看线性投影的效果，再用 t-SNE，对比簇的清晰度。
**Digits** (64-D handwritten digits, a mini MNIST). We compare PCA's linear projection with t-SNE on cluster clarity.

> 常规做法：**先用 PCA 把维度预降到 ~30 维**（去噪 + 加速），再喂给 t-SNE。
> Common practice: **pre-reduce to ~30 dims with PCA** (denoise + speed up) before feeding t-SNE.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
sns.set_theme(style="whitegrid")

digits = load_digits()
X, y = digits.data, digits.target
print(f"Digits: {X.shape}, 10 类 classes 0-9")

# 先 PCA 预降到 30 维(降噪+加速), 再做 t-SNE / PCA pre-reduction then t-SNE
X30 = PCA(n_components=30, random_state=0).fit_transform(X)
t = time.perf_counter()
# perplexity=30 ≈ 每个点考虑 30 个有效邻居; init='pca' 用 PCA 结果当初始布局(更稳)
Z = TSNE(n_components=2, perplexity=30, init="pca", random_state=0).fit_transform(X30)
print(f"t-SNE 用时 took {time.perf_counter()-t:.1f}s")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
p2 = PCA(2, random_state=0).fit_transform(X)
axes[0].scatter(p2[:,0], p2[:,1], c=y, cmap="tab10", s=10)
axes[0].set_title("PCA 2D: 数字簇大量重叠(线性投影看不清) / clusters overlap")
sc1 = axes[1].scatter(Z[:,0], Z[:,1], c=y, cmap="tab10", s=10)
axes[1].set_title("t-SNE 2D: 10 个数字簇清晰分离 / 10 digit clusters cleanly separated")
plt.colorbar(sc1, ax=axes[1], label="digit"); plt.tight_layout(); plt.show()
print("t-SNE 把同类数字聚成清晰簇 → 高维数据探索/可视化标准工具 / the standard viz tool")


<a id="4"></a>
## 4. perplexity 的影响 ⭐ / Effect of Perplexity

**perplexity** ≈ 每个点考虑的"有效邻居数"（常用 5–50）。太小 → 只看极近的几个邻居，结果碎、噪声大；太大 → 邻域过宽，簇糊在一起。它是 t-SNE 最重要的旋钮，**必须多试几个值**。
**perplexity** ≈ the "effective number of neighbors" each point considers (typically 5–50). Too small → only a few very close neighbors, fragmented and noisy; too large → neighborhoods too wide, clusters blur together. It's t-SNE's most important knob — **always try several values**.


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, perp in zip(axes, [5, 30, 50, 100]):        # 试 4 个 perplexity 值
    Zp = TSNE(2, perplexity=perp, init="pca", random_state=0).fit_transform(X30)
    ax.scatter(Zp[:,0], Zp[:,1], c=y, cmap="tab10", s=6)
    ax.set_title(f"perplexity={perp}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("perplexity: 太小(5)碎裂噪声大; 适中(30-50)清晰; 太大(100)开始糊在一起 / small=fragmented, large=blurred")
plt.tight_layout(); plt.show()
print("perplexity 是核心超参(有效邻居数, 常 5-50); 不同值结果差异大, 应多试")
print("perplexity is the key knob (effective #neighbors, usually 5-50); try several.")


<a id="5"></a>
## 5. 误区：什么不可信 ⭐ / What NOT to Trust

t-SNE 极易被误读（面试高频）。**不要**从 t-SNE 图里读出这些结论：
t-SNE is easily misread (frequently asked). **Do NOT** read these off a t-SNE plot:
- ❌ **簇的大小 / cluster size**：t-SNE 会把稠密的簇放大、稀疏的簇缩小，簇的面积**毫无意义**。
  t-SNE inflates dense clusters and shrinks sparse ones; cluster area is **meaningless**.
- ❌ **簇间距离 / between-cluster distance**：两簇离得远≠真的更不相似；**全局距离不保真**（它只保局部邻域）。
  Two clusters far apart ≠ truly more dissimilar; **global distances aren't preserved** (only local neighborhoods are).
- ❌ **空白间隙 / gaps**：间隙宽窄不代表什么。
  The width of gaps means nothing.
- ⚠️ **随机性 / stochastic**：每次运行（不同 seed）布局都不同；形状会变。
  Each run (different seed) gives a different layout; shapes change.
- ⚠️ **不能 transform 新点 / no transform**：t-SNE 没有可复用的映射函数（每次都要把整批数据重算）→ 不能放进生产推理 pipeline。
  t-SNE has no reusable mapping (it recomputes the whole batch each time) → can't go into a production inference pipeline.

✅ **能信的 / What you CAN trust**：哪些点彼此是近邻（局部结构）、数据有没有分成几团。它是**探索/可视化**工具，不是建模/降维特征工具。
✅ Which points are neighbors (local structure) and whether the data splits into groups. It's an **exploration/visualization** tool, not a feature-engineering one.


In [ ]:
# 演示: 换随机种子 → 布局变了, 但簇结构不变 / different seeds → different layout, same clusters
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, seed in zip(axes, [0, 1, 2]):
    # init='random' + 不同 random_state, 看布局怎么随机变化
    Zs = TSNE(2, perplexity=30, init="random", random_state=seed).fit_transform(X30)
    ax.scatter(Zs[:,0], Zs[:,1], c=y, cmap="tab10", s=6)
    ax.set_title(f"random_state={seed}"); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("不同随机种子: 簇的相对位置/朝向都变 → 别解读簇间距离与全局布局 / global layout is arbitrary")
plt.tight_layout(); plt.show()
print("簇内成员稳定, 但簇的相对位置/旋转随机变 → 全局几何不可信")
print("Cluster membership is stable, but their relative positions/rotations are arbitrary.")


<a id="6"></a>
## 6. 小结 / Summary

```
t-SNE: 高维相似度 p_ij(高斯, 由perplexity定带宽) ↔ 低维 q_ij(t分布) 匹配, 最小化 KL(P‖Q)
低维用 t 分布(重尾)→ 解决拥挤问题, 给中距离点对留空间, 簇分得开
perplexity(有效邻居数, 5-50)是核心超参, 多试几个
不可信: 簇大小/簇间距离/全局结构/间隙; 随机(每次不同); 不能 transform 新点
能信: 局部邻域 + 是否分团; 是探索/可视化工具, 非降维特征工具
常规: 先 PCA 预降到 ~30-50 维再 t-SNE(降噪+加速)
```

### 💡 面试速查 / Interview cheat-sheet
1. **原理**：高维高斯相似度 vs 低维 t 分布，最小化 KL 散度。
   High-dim Gaussian similarity vs low-dim t, minimize KL divergence.
2. **t 分布重尾解决拥挤问题**（高维体积 >> 2D）。
   The heavy-tailed t fixes crowding (high-dim volume ≫ 2-D).
3. **perplexity** = 有效邻居数，最关键超参。
   perplexity = effective #neighbors, the key hyperparameter.
4. **不可信**：簇大小/簇间距离/全局几何；**不能 transform** 新点。
   Don't trust cluster size/distance/global geometry; can't transform new points.
5. **vs PCA**：非线性保局部 vs 线性保全局方差；t-SNE 慢、仅可视化。
   vs PCA: nonlinear local vs linear global variance; t-SNE is slow and viz-only.

### 下一节 / Next
**6.13 UMAP**——和 t-SNE 一样保局部、可视化强，但**更快、能 transform 新数据、更好地保留一些全局结构**，已逐渐成为新首选。
**6.13 UMAP** — like t-SNE preserves locality and visualizes well, but is **faster, can transform new data, and keeps more global structure**; increasingly the new default.
